In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.2,
)

In [3]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [4]:
llm_with_tools = model.bind_tools([multiply])

In [5]:
llm_with_tools.invoke('Hi how are you')

# No use for tool call so its not mentioned in output

c:\Users\kumar\OneDrive\Desktop\Resume\LangChain\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3610: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AIMessage(content=[{'type': 'text', 'text': "Hello! I'm doing well, thank you for asking. How can I help you today?", 'extras': {'signature': 'EmAKXgFpFH0TSfdsjGh19yqdj4BIaPQR1/TUG/BEV2WHDOCacAEY8aG5MKyIbIJKJXw64bh2uPJNZ8eZPPLKHkKgmtlA++LxwvOJIRDn906PbVl5eeM5Eqdx2r+OTnNBw+4='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1cfbc-00d4-7d03-974c-9c86c19b9822-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 64, 'output_tokens': 20, 'total_tokens': 84, 'input_token_details': {'cache_read': 0}})

In [10]:
llm_with_tools.invoke("What is the product of 3 and 6?").tool_calls

# here the content is empty unlike last time and there is a new section called tool_call. Its a list of tools but here we have only one.

result = llm_with_tools.invoke("What is the product of 3 and 6?") # as we see,  a list of tools
print(result.tool_calls)

c:\Users\kumar\OneDrive\Desktop\Resume\LangChain\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3610: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\kumar\OneDrive\Desktop\Resume\LangChain\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3610: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'name': 'multiply', 'args': {'a': 3, 'b': 6}, 'id': 'call_9209658138159882873', 'type': 'tool_call'}]


In [11]:
multiply.invoke(result.tool_calls[0])

# Tool message is better than just message so we just call the tool with the tool_call object and it will return the output of the tool.

ToolMessage(content='18', name='multiply', tool_call_id='call_9209658138159882873')

In [ ]:
Now we wanna mantain sort of a history for better use'

We will create a human message and then add it to a list and then the AI message models reponds us tellling us which tool to use and then we will call the tool and then we will add the tool response to the list and then we will call the AI model again with the updated list and then it will give us the final response.

In [14]:
query = HumanMessage('can you multiply 3 with 1000')

messages = [query]

result = llm_with_tools.invoke(messages)
messages.append(result)

c:\Users\kumar\OneDrive\Desktop\Resume\LangChain\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3610: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [15]:
tool_result = multiply.invoke(result.tool_calls[0])
# this is the tool message that we got from the tool call and we will add it to the messages list so that we can use it in the next call to the AI model.

messages.append(tool_result)

In [16]:
print(messages)

# it has all three messages, the human message, the AI message and the tool message. Now we will call the AI model again with the updated list of messages and it will give us the final response.

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"b": 1000, "a": 3}'}, '__gemini_function_call_thought_signatures__': {'call_612926': 'El4KXAERTTIPxOVcNgX1vNQ+qHgBINBKXTNeZqb71KWyp4S7+wOyM4Ct7O7Q3zrioercCgwXs+3gKCLcmBZfkQ0azGFEyrJAKEMU02RvXU6WSBV386CrsqSC986wSgx1'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a067b7-7d5f-7d93-a91e-de1cbbe9daf9-0', tool_calls=[{'name': 'multiply', 'args': {'b': 1000, 'a': 3}, 'id': 'call_612926', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 71, 'output_tokens': 19, 'total_tokens': 90, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='3000', name='multiply', tool_call_id='call_612926')]


In [17]:
llm_with_tools.invoke(messages).content

c:\Users\kumar\OneDrive\Desktop\Resume\LangChain\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3610: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text',
  'text': '3 multiplied by 1000 is 3000.',
  'extras': {'signature': 'El4KXAERTTIPaznhwDkclHAUyMpxh5WVqQRA++ol0x32BhjTucaq2XibHoUzUq1jr54KuBllcG2OiGX3l08xDY4D0VURDERNQl6Raikj0gJTmjzpX8ywrw09cYSZZNFt'}}]

## Exchange Rate Between Currencies

In [87]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def conversion_rate(base_currency:str, target_currency:str) -> float:
    """
    This function fetches the currency conversion factor between a given base currency and a target currency
    """
    link= f"https://v6.exchangerate-api.com/v6/967ab55646c462eacda33415/pair/{base_currency}/{target_currency}"

    response = requests.get(link)
    return response.json()["conversion_rate"]

@tool
def convert(base_currency_value:int, conversion_factor: Annotated[float, InjectedToolArg])-> float:
    '''
    given a currency conversion rate this function calculates the target currency value from a given base currency value
    '''
    return base_currency_value * conversion_factor

In [88]:
# conversion_value = conversion_rate.invoke({'base_currency':'USD','target_currency':'INR'})
# conversion_rate.invoke({'base_currency':'USD','target_currency':'INR'})

convert.invoke({'base_currency_value':10, 'conversion_factor':conversion_rate.invoke({'base_currency':'USD','target_currency':'INR'})})

948.986

In [89]:
convert.invoke({'base_currency_value':10, 'conversion_factor':85.16})

851.5999999999999

In [90]:
llm_with_tools = model.bind_tools([conversion_rate, convert])

In [91]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [93]:
ai_message = llm_with_tools.invoke(messages)
print(ai_message)

c:\Users\kumar\OneDrive\Desktop\Resume\LangChain\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3610: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


content=[] additional_kwargs={'function_call': {'name': 'conversion_rate', 'arguments': '{"target_currency": "USD", "base_currency": "INR"}'}, '__gemini_function_call_thought_signatures__': {'call_958236': 'El4KXAERTTIPu1+AJAjrcHQmNAJB3c7kVTZZrcb6RE8PNE5p2MkryL7n+lfuftDOBsmO6ERYv8ZfRt1CD1hVyo7vKqHBSovPLEczIcBRPBSBimF8WnSt9GrMYfRrYS0B'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a067ed-a4d7-7a82-bae2-085b74c4f056-0' tool_calls=[{'name': 'conversion_rate', 'args': {'target_currency': 'USD', 'base_currency': 'INR'}, 'id': 'call_958236', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 159, 'output_tokens': 26, 'total_tokens': 185, 'input_token_details': {'cache_read': 0}}


In [ ]:
messages.append(ai_message)

In [98]:
ai_message.tool_calls

[{'name': 'conversion_rate',
  'args': {'target_currency': 'USD', 'base_currency': 'INR'},
  'id': 'call_958236',
  'type': 'tool_call'}]

In [76]:
for tool_call in ai_message.tool_calls:
    print(tool_call)

{'name': 'conversion_rate', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': 'call_1044266', 'type': 'tool_call'}


In [81]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = conversion_rate.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [82]:
llm_with_tools.invoke(messages).content

ValueError: Model 'gemini-3.5-flash-lite' does not support model prefilling. The final request turn must be a user message or a function response.